In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load original cleaned data (has all features)
df_original = pd.read_csv('telco_customer_churn_cleaned.csv')

# Load churn probabilities from Phase 4
df_churn = pd.read_csv('telco_churn_probability.csv')

# Merge: churn probability + original features
clv_dataset = df_original.merge(
    df_churn[['customerID', 'churn_probability']], 
    on='customerID', 
    how='inner'
)

print(f"\n✓ Loaded dataset with {clv_dataset.shape[0]:,} customers")
print(f"✓ Columns: {clv_dataset.shape[1]}")
print(f"\nColumn names:")
for col in clv_dataset.columns:
    print(f"  - {col}")


✓ Loaded dataset with 5,636 customers
✓ Columns: 32

Column names:
  - customerID
  - SeniorCitizen
  - tenure
  - MonthlyCharges
  - TotalCharges
  - churn_flag
  - partner_flag
  - dependents_flag
  - phoneservice_flag
  - paperlessbilling_flag
  - contract_ordinal
  - gender_Male
  - internet_Fiber optic
  - internet_No
  - multiplelines_No phone service
  - multiplelines_Yes
  - onlinesecurity_No internet service
  - onlinesecurity_Yes
  - onlinebackup_No internet service
  - onlinebackup_Yes
  - deviceprotection_No internet service
  - deviceprotection_Yes
  - techsupport_No internet service
  - techsupport_Yes
  - streamingtv_No internet service
  - streamingtv_Yes
  - streamingmovies_No internet service
  - streamingmovies_Yes
  - payment_Credit card (automatic)
  - payment_Electronic check
  - payment_Mailed check
  - churn_probability


In [20]:
print("Columns in df_original:")
print(df_original.columns.tolist())
print("\n\nColumns in clv_dataset after merge:")
print(clv_dataset.columns.tolist())

Columns in df_original:
['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'churn_flag', 'partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag', 'contract_ordinal', 'gender_Male', 'internet_Fiber optic', 'internet_No', 'multiplelines_No phone service', 'multiplelines_Yes', 'onlinesecurity_No internet service', 'onlinesecurity_Yes', 'onlinebackup_No internet service', 'onlinebackup_Yes', 'deviceprotection_No internet service', 'deviceprotection_Yes', 'techsupport_No internet service', 'techsupport_Yes', 'streamingtv_No internet service', 'streamingtv_Yes', 'streamingmovies_No internet service', 'streamingmovies_Yes', 'payment_Credit card (automatic)', 'payment_Electronic check', 'payment_Mailed check']


Columns in clv_dataset after merge:
['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'churn_flag', 'partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag', 'contract_ordinal', 'gender_Ma

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 2: CREATE NUM_SERVICES (count of adopted services)
# ══════════════════════════════════════════════════════════════════════════
# Service columns: phoneservice, onlinesecurity, onlinebackup, deviceprotection,
#                  techsupport, streamingtv, streamingmovies

service_cols = ['phoneservice_flag', 'onlinesecurity_Yes', 'onlinebackup_Yes', 
                'deviceprotection_Yes', 'techsupport_Yes', 'streamingtv_Yes', 'streamingmovies_Yes']

clv_dataset['num_services'] = clv_dataset[service_cols].sum(axis=1)

print(f"\nCreated num_services (count of adopted services)")
print(f"  Range: {clv_dataset['num_services'].min():.0f} - {clv_dataset['num_services'].max():.0f}")
print(f"  Mean: {clv_dataset['num_services'].mean():.2f}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2B: IDENTIFY CLV COMPONENTS FROM AVAILABLE DATA
# ══════════════════════════════════════════════════════════════════════════

print(f"\n1. REVENUE SIGNALS:")
revenue_cols = ['MonthlyCharges', 'TotalCharges', 'tenure']
for col in revenue_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: ${clv_dataset[col].mean():.2f})")
    else:
        print(f"  {col:20s} - Not found")

# Engagement/Stickiness Components

print(f"\n2. ENGAGEMENT SIGNALS (Stickiness):")

engagement_cols = ['num_services', 'contract_ordinal']

for col in engagement_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: {clv_dataset[col].mean():.3f})")
    else:
        print(f"  {col:20s} - Not found")

# Risk Components

print(f"\n3. RETENTION/RISK SIGNALS:")

risk_cols = ['churn_probability']

for col in risk_cols:
    if col in clv_dataset.columns:
        print(f"  {col:20s} - Available (Mean: {clv_dataset[col].mean():.3f})")
    else:
        print(f"  {col:20s} - Not found")

print(f"\n✓ All key components available for CLV calculation")


Created num_services (count of adopted services)
  Range: 0 - 7
  Mean: 2.59

1. REVENUE SIGNALS:
  MonthlyCharges       - Available (Mean: $61.97)
  TotalCharges         - Available (Mean: $1555.61)
  tenure               - Available (Mean: $23.45)

2. ENGAGEMENT SIGNALS (Stickiness):
  num_services         - Available (Mean: 2.588)
  contract_ordinal     - Available (Mean: 0.457)

3. RETENTION/RISK SIGNALS:
  churn_probability    - Available (Mean: 0.312)

✓ All key components available for CLV calculation


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 3: CREATE DERIVED FEATURES FOR CLV MODELING
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("CREATING DERIVED CLV FEATURES")
print("="*60)

# Copy dataset for feature engineering
clv_data = clv_dataset.copy()

# ──────────────────────────────────────────────────────────────────────────
# Feature 1: Lifetime Spend (past revenue = what they've already paid)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Shows customer investment to date (higher = more valuable relationship)
clv_data['lifetime_spend'] = clv_data['TotalCharges']

print(f"\n1. LIFETIME_SPEND (Total Charges to Date)")
print(f"   What: Historical cumulative revenue from this customer")
print(f"   Why: Reflects total relationship investment")
print(f"   Range: ${clv_data['lifetime_spend'].min():.2f} - ${clv_data['lifetime_spend'].max():.2f}")
print(f"   Mean: ${clv_data['lifetime_spend'].mean():.2f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 2: Average Monthly Spend (current revenue rate)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Shows current revenue stream (annualize to get future potential)
clv_data['avg_monthly_spend'] = clv_data['MonthlyCharges']

print(f"\n2. AVG_MONTHLY_SPEND (Monthly Charges)")
print(f"   What: Current monthly revenue from this customer")
print(f"   Why: Basis for projecting future revenue")
print(f"   Range: ${clv_data['avg_monthly_spend'].min():.2f} - ${clv_data['avg_monthly_spend'].max():.2f}")
print(f"   Mean: ${clv_data['avg_monthly_spend'].mean():.2f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 3: Retention Rate (from churn probability)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Use churn probability to weight future revenue expectations
clv_data['retention_rate'] = 1 - clv_data['churn_probability']

print(f"\n3. RETENTION_RATE (1 - Churn Probability)")
print(f"   What: Probability customer stays (complement of churn risk)")
print(f"   Why: Use to weight future revenue expectations")
print(f"   Range: {clv_data['retention_rate'].min():.3f} - {clv_data['retention_rate'].max():.3f}")
print(f"   Mean: {clv_data['retention_rate'].mean():.3f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 4: Revenue Stability (monthly vs total ratio)
# ──────────────────────────────────────────────────────────────────────────
# WHY: High ratio = recent high spending; Low ratio = already paid off early years
clv_data['revenue_stability'] = clv_data['MonthlyCharges'] / (clv_data['TotalCharges'] + 1)

print(f"\n4. REVENUE_STABILITY (Monthly / Total Charges)")
print(f"   What: Ratio of current to historical spending")
print(f"   Why: Indicates if customer spending is increasing or declining")
print(f"   Range: {clv_data['revenue_stability'].min():.4f} - {clv_data['revenue_stability'].max():.4f}")
print(f"   Mean: {clv_data['revenue_stability'].mean():.4f}")

# ──────────────────────────────────────────────────────────────────────────
# Feature 5: Engagement Level (tenure + num_services normalized)
# ──────────────────────────────────────────────────────────────────────────
# WHY: Customers with more services are more sticky (harder to leave)
tenure_norm = clv_data['tenure'] / clv_data['tenure'].max()
services_norm = clv_data['num_services'] / clv_data['num_services'].max()
clv_data['engagement_level'] = (tenure_norm + services_norm) / 2

print(f"\n5. ENGAGEMENT_LEVEL (Tenure + Service Count)")
print(f"   What: Combined tenure and service adoption score (0-1)")
print(f"   Why: Higher engagement = stickier customers = lower churn")
print(f"   Range: {clv_data['engagement_level'].min():.3f} - {clv_data['engagement_level'].max():.3f}")
print(f"   Mean: {clv_data['engagement_level'].mean():.3f}")

print(f"\n✓ All CLV features created successfully")


CREATING DERIVED CLV FEATURES

1. LIFETIME_SPEND (Total Charges to Date)
   What: Historical cumulative revenue from this customer
   Why: Reflects total relationship investment
   Range: $18.80 - $7049.50
   Mean: $1555.61

2. AVG_MONTHLY_SPEND (Monthly Charges)
   What: Current monthly revenue from this customer
   Why: Basis for projecting future revenue
   Range: $18.25 - $117.45
   Mean: $61.97

3. RETENTION_RATE (1 - Churn Probability)
   What: Probability customer stays (complement of churn risk)
   Why: Use to weight future revenue expectations
   Range: 0.078 - 0.994
   Mean: 0.688

4. REVENUE_STABILITY (Monthly / Total Charges)
   What: Ratio of current to historical spending
   Why: Indicates if customer spending is increasing or declining
   Range: 0.0155 - 0.9903
   Mean: 0.1915

5. ENGAGEMENT_LEVEL (Tenure + Service Count)
   What: Combined tenure and service adoption score (0-1)
   Why: Higher engagement = stickier customers = lower churn
   Range: 0.008 - 1.000
   Mean

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 4: EXPLAIN THE CLV CALCULATION LOGIC
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("WHY THESE FEATURES MATTER FOR CLV")
print("="*60)

explanation = """
┌─────────────────────────────────────────────────────────────┐
│ KEY INSIGHT: Past Revenue ≠ Future Value                   │
└─────────────────────────────────────────────────────────────┘

TRADITIONAL MISTAKE:
  "Customer paid $3,000 total → CLV = $3,000"
  WRONG! This ignores future churn risk

CORRECT APPROACH (What we're doing):
  1. How much has customer spent? ($3,000) → HISTORICAL value
  2. How much do they spend per month? ($100) → REVENUE RATE
  3. How likely are they to stay? (80%) → RETENTION probability
  4. Therefore: Expected future value = $100/month × 12 × 0.8 = ~$960

┌─────────────────────────────────────────────────────────────┐
│ WHY ENGAGEMENT MATTERS FOR CLV                              │
└─────────────────────────────────────────────────────────────┘

CUSTOMER A: Trial Period Customer
  - tenure: 2 months
  - num_services: 1 (phone only)
  - monthly_charges: $20
  - churn_probability: 0.50 (HIGH RISK!)
  
  Traditional CLV: $20 × 2 = $40
  Smart CLV: $20/month × 12 × 0.5 = $120 (but needs retention!)
  Action: INVEST in retention (cross-sell, loyalty program)

CUSTOMER B: Established, Multi-Service
  - tenure: 36 months
  - num_services: 4 (phone + internet + TV + security)
  - monthly_charges: $150
  - churn_probability: 0.05 (LOW RISK!)
  
  Traditional CLV: $150 × 36 = $5,400
  Smart CLV: $150/month × 12 × 0.95 = $1,710/year + expansion potential
  Action: PROTECT (maintain quality, upsell premium services)

┌─────────────────────────────────────────────────────────────┐
│ FEATURES THAT PREDICT HIGH CLV                              │
└─────────────────────────────────────────────────────────────┘

✓ HIGH engagement_level (long tenure + many services)
  → Hard to switch, integrated into daily life
  
✓ HIGH monthly_charges (revenue per user)
  → Already invested in premium services
  
✓ LOW churn_probability (high retention_rate)
  → Stable, proven loyalty
  
✓ HIGH tenure_normalized (years of relationship)
  → Habit formation, switching costs

✗ LOW features indicate churn risk
  → Trial period customers need intervention
  → New, single-service customers vulnerable
"""

print(explanation)

print(f"\n✓ CLV framework defined and ready for Phase 5 Part 2")
print(f"  Next: Build actual CLV score combining these features")


WHY THESE FEATURES MATTER FOR CLV

┌─────────────────────────────────────────────────────────────┐
│ KEY INSIGHT: Past Revenue ≠ Future Value                   │
└─────────────────────────────────────────────────────────────┘

TRADITIONAL MISTAKE:
  "Customer paid $3,000 total → CLV = $3,000"
  WRONG! This ignores future churn risk

CORRECT APPROACH (What we're doing):
  1. How much has customer spent? ($3,000) → HISTORICAL value
  2. How much do they spend per month? ($100) → REVENUE RATE
  3. How likely are they to stay? (80%) → RETENTION probability
  4. Therefore: Expected future value = $100/month × 12 × 0.8 = ~$960

┌─────────────────────────────────────────────────────────────┐
│ WHY ENGAGEMENT MATTERS FOR CLV                              │
└─────────────────────────────────────────────────────────────┘

CUSTOMER A: Trial Period Customer
  - tenure: 2 months
  - num_services: 1 (phone only)
  - monthly_charges: $20
  - churn_probability: 0.50 (HIGH RISK!)

  Traditional CLV: $2

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 5: SUMMARY STATISTICS OF CLV FEATURES
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("CLV DATASET READY FOR PHASE 5 PART 2")
print("="*60)

print(f"\nDataset shape: {clv_data.shape[0]:,} customers × {clv_data.shape[1]} features")

print(f"\nNew CLV Features Created:")
clv_features = ['lifetime_spend', 'avg_monthly_spend', 'retention_rate', 
                'revenue_stability', 'engagement_level']
for feat in clv_features:
    if feat in clv_data.columns:
        print(f"  ✓ {feat}")

print(f"\nFeature Statistics:")
feature_stats = clv_data[[col for col in clv_features if col in clv_data.columns]].describe()
print(feature_stats.to_string())

print(f"\n Phase 5 Part 1 Complete")
print(f"  Dataset ready for CLV scoring in Phase 5 Part 2")
print(f"  Variables available: clv_data (with all features)")


CLV DATASET READY FOR PHASE 5 PART 2

Dataset shape: 5,636 customers × 38 features

New CLV Features Created:
  ✓ lifetime_spend
  ✓ avg_monthly_spend
  ✓ retention_rate
  ✓ revenue_stability
  ✓ engagement_level

Feature Statistics:
       lifetime_spend  avg_monthly_spend  retention_rate  revenue_stability  engagement_level
count     5636.000000        5636.000000     5636.000000        5636.000000       5636.000000
mean      1555.610690          61.967912        0.687515           0.191527          0.380292
std       1605.453955          28.971378        0.251529           0.298371          0.227863
min         18.800000          18.250000        0.077906           0.015514          0.008333
25%        265.337500          31.175000        0.490532           0.025272          0.179762
50%        940.075000          69.000000        0.743547           0.050433          0.351190
75%       2448.087500          85.700000        0.918269           0.172137          0.546429
max       704